# 04 · Método de IA: VGGT

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StevenFAP1815/project-reconstruction-3d/blob/main/notebooks/04_ai_vggt.ipynb)

Reconstrucción del mismo objeto y con las mismas fotos que COLMAP, pero con una
red neuronal (criterio 7 de la rúbrica).

**Principio del método.** VGGT (*Visual Geometry Grounded Transformer*, Wang et
al., CVPR 2025) recibe un conjunto de fotos y predice en **una sola pasada** la
cámara de cada foto (posición, orientación y focal), un mapa de profundidad por
foto y la confianza de cada píxel. No detecta puntos SIFT ni hace bundle
adjustment: un *transformer* (la arquitectura de los modelos de lenguaje,
aplicada a parches de imagen) compara todas las fotos entre sí mediante
atención, y lo que aprendió con millones de escenas sustituye a la geometría
explícita de COLMAP.

| | COLMAP (02 y 03) | VGGT (este notebook) |
|---|---|---|
| Correspondencias | SIFT + RANSAC, explícitas | Implícitas (atención entre parches) |
| Cámaras | Registro incremental + bundle adjustment | Predichas directamente |
| Profundidad | PatchMatch Stereo | Predicha directamente (518 px) |
| Depende de la textura | Mucho | Menos (usa conocimiento previo de formas) |
| Coste | Minutos a decenas de minutos | Segundos, pero con mucha memoria de GPU |

**Diferencias de condiciones que se deben declarar en el informe:**

- VGGT recibe `N_FRAMES` fotos (repartidas de forma uniforme entre las nítidas),
  no todas, porque la memoria de la GPU crece con el número de fotos.
- VGGT trabaja a 518 px de lado; COLMAP, a 1600 px (MVS) y a resolución
  completa (SfM).
- La escala y el error dimensional se calculan exactamente igual que en `03`.

**Entorno:** GPU. El modelo (`facebook/VGGT-1B`, unos 5 GB) se descarga de
Hugging Face en la primera ejecución. Licencia de los pesos: no comercial (uso
académico permitido).

## 1. Entorno

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/StevenFAP1815/project-reconstruction-3d.git"
REPO_DIR = Path("/content/project-reconstruction-3d")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

# Python importa cada módulo una sola vez por sesión. Si esta sesión ya había
# cargado recon3d (una ejecución anterior), se descarta para que el siguiente
# import lea el código recién descargado y no una versión antigua en memoria.
for module in [name for name in sys.modules if name == "recon3d" or name.startswith("recon3d.")]:
    del sys.modules[module]

COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("Código del repositorio en el commit", COMMIT)

VGGT se instala desde su repositorio, fijado en un commit concreto para que la
corrida sea reproducible. Se instala **sin sus dependencias** (`--no-deps`):
VGGT declara `numpy<2`, y reinstalar numpy 1.x rompería OpenCV y el resto del
entorno de Colab (que ya trae torch, numpy, Pillow, safetensors y
huggingface_hub). Solo se añade `einops`.

In [ ]:
VGGT_COMMIT = "a288dd0f14786c93483e45524328726ab7b1b4ce"
%pip install -q --no-deps "git+https://github.com/facebookresearch/vggt@{VGGT_COMMIT}"
%pip install -q einops

## 2. Parámetros

In [ ]:
OBJECT = "caja-carton-leche"   # caja-carton-leche | canasto-flores | audifonos | colador-cocina | vaso-vidrio
METHOD = "vggt"
SEED = 42
N_FRAMES = 60                  # fotos que recibe VGGT (más = más memoria de GPU)
BLUR_RATIO = 0.35              # mismo filtro de fotos borrosas que en 02
CONF_PERCENTILE = 50           # se descarta el 50 % de píxeles menos fiables
PRECISION = "auto"             # "auto" (float16 en T4) o "float32" si salen NaN
MAX_POINTS = 2_000_000         # tope de puntos para la limpieza (memoria y tiempo)

# Limpieza: los mismos valores que en 03, para comparar en igualdad de condiciones.
CROP_RATIO = 0.6
PLANE_RATIO = 0.01
VOXEL_RATIO = 0.01
MIN_POINTS_PER_VOXEL = 3

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import torch

from recon3d.dataset import extract_object_zip
from recon3d.paths import get_paths
from recon3d.preprocessing import flag_blurry, laplacian_variance
from recon3d.reproducibility import set_seed
from recon3d.results import environment_info, write_meta
from recon3d.vggt_tools import select_frames

set_seed(SEED)
paths = get_paths()
paths.create()
start_total = time.perf_counter()

LOCAL_IMAGES = Path("/content/data") / OBJECT
images = extract_object_zip(paths.data_zip(OBJECT), LOCAL_IMAGES)
sharpness = {path.name: laplacian_variance(path) for path in images}
blurry = flag_blurry(sharpness, ratio=BLUR_RATIO)
sharp = [path for path in images if path.name not in blurry]
frames = select_frames(sharp, N_FRAMES)

print(f"{OBJECT}: {len(images)} fotos, {len(blurry)} borrosas, {len(frames)} para VGGT")
print("torch", torch.__version__, "| GPU:", environment_info()["gpu"])

## 3. Inferencia

Toda la reconstrucción ocurre en la llamada `model(images)`. Las fotos se
reducen a 518 px de lado mayor y se rellenan con blanco hasta un cuadrado
(`mode="pad"`), así no se recorta ninguna parte del objeto.

In [ ]:
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri

device = "cuda"
if PRECISION == "float32":
    dtype = torch.float32
else:  # bfloat16 solo existe en GPU Ampere (8.0+); la T4 es 7.5 y usa float16
    dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

start = time.perf_counter()
model = VGGT.from_pretrained("facebook/VGGT-1B").to(device).eval()
load_s = time.perf_counter() - start
images_tensor = load_and_preprocess_images([str(p) for p in frames], mode="pad").to(device)

torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()
with torch.no_grad(), torch.autocast("cuda", dtype=dtype, enabled=dtype != torch.float32):
    predictions = model(images_tensor)
torch.cuda.synchronize()
inference_s = time.perf_counter() - start
peak_gpu_gb = torch.cuda.max_memory_allocated() / 1e9

extrinsic, intrinsic = pose_encoding_to_extri_intri(predictions["pose_enc"].float(), images_tensor.shape[-2:])
extrinsic = extrinsic[0].cpu().numpy()
intrinsic = intrinsic[0].cpu().numpy()
depth = predictions["depth"][0, ..., 0].float().cpu().numpy()       # (S, H, W)
confidence = predictions["depth_conf"][0].float().cpu().numpy()     # (S, H, W)
colors_all = (images_tensor.permute(0, 2, 3, 1).float().cpu().numpy() * 255).astype(np.uint8)

print(f"Carga del modelo: {load_s:.1f} s | inferencia de {len(frames)} fotos: {inference_s:.1f} s")
print(f"Memoria máxima de GPU: {peak_gpu_gb:.2f} GB | precisión: {dtype}")
print("Valores no finitos en la profundidad:", int((~np.isfinite(depth)).sum()))

In [ ]:
import matplotlib.pyplot as plt

from recon3d.vggt_tools import content_box
from PIL import Image

x0, y0, w, h = content_box(Image.open(frames[0]).size)
show = [0, len(frames) // 3, 2 * len(frames) // 3]
fig, axes = plt.subplots(3, len(show), figsize=(4 * len(show), 14))
for col, i in enumerate(show):
    crop = (slice(y0, y0 + h), slice(x0, x0 + w))
    axes[0, col].imshow(colors_all[i][crop])
    axes[0, col].set_title(frames[i].name, fontsize=8)
    axes[1, col].imshow(depth[i][crop], cmap="turbo")
    axes[1, col].set_title("profundidad predicha")
    axes[2, col].imshow(confidence[i][crop], cmap="magma")
    axes[2, col].set_title("confianza")
for ax in axes.ravel():
    ax.axis("off")
plt.show()

## 4. Cámaras predichas

Se comparan las cámaras de VGGT con la calibración del tablero: su focal debe
coincidir con la calibrada, convertida a la resolución de VGGT. Es una prueba
de que la red "entiende" la geometría de la cámara sin haberla visto antes.

In [ ]:
from recon3d.geometry import trajectory_frame
from recon3d.vggt_tools import camera_rays, expected_focal

calibration = json.loads((paths.calibration_results / "calibration.json").read_text(encoding="utf-8"))
fx_ref, fy_ref = expected_focal((calibration["fx"], calibration["fy"]), Image.open(frames[0]).size)
fx_vggt, fy_vggt = np.median(intrinsic[:, 0, 0]), np.median(intrinsic[:, 1, 1])
focal_table = pd.DataFrame({
    "calibración (518 px)": [fx_ref, fy_ref],
    "VGGT (mediana)": [fx_vggt, fy_vggt],
}, index=["fx", "fy"])
focal_table["diferencia %"] = 100 * (focal_table["VGGT (mediana)"] / focal_table["calibración (518 px)"] - 1)
display(focal_table.round(2))

centers, directions = camera_rays(extrinsic)
coords_cam, orbit_axes = trajectory_frame(centers)
fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(coords_cam[:, 0], coords_cam[:, 1], c=np.arange(len(centers)), cmap="viridis")
ax.set_aspect("equal")
ax.set_title(f"Trayectoria predicha por VGGT ({len(centers)} cámaras, color = orden)")
fig.colorbar(sc, ax=ax, label="índice de foto")
plt.show()

## 5. Nube de puntos, limpieza, escala y error dimensional

Cada píxel con profundidad se retroproyecta al espacio 3D con su cámara
(`recon3d.vggt_tools.unproject_depth`). Se descartan los píxeles de relleno y
los de baja confianza. Después se aplica **la misma limpieza, la misma caja
envolvente y las mismas medidas que en `03`**.

In [ ]:
from recon3d.pointcloud import clean_object_cloud, look_at_point
from recon3d.vggt_tools import confidence_mask, unproject_depth

world = unproject_depth(depth, extrinsic, intrinsic)                 # (S, H, W, 3)
inside = np.zeros(depth.shape, dtype=bool)
inside[:, y0:y0 + h, x0:x0 + w] = True                                 # fuera = relleno blanco
valid = inside & np.isfinite(world).all(axis=-1) & (depth > 0)
# El umbral de confianza se calcula solo sobre los píxeles de la foto real.
valid[valid] = confidence_mask(confidence[valid], CONF_PERCENTILE)
points, colors = world[valid], colors_all[valid]
if len(points) > MAX_POINTS:
    keep = np.random.default_rng(SEED).choice(len(points), MAX_POINTS, replace=False)
    points, colors = points[keep], colors[keep]

up = orbit_axes[2]
if np.mean((centers - look_at_point(centers, directions)) @ up) < 0:
    up = -up
mask, clean_report = clean_object_cloud(
    points, centers, directions, up,
    crop_ratio=CROP_RATIO, plane_ratio=PLANE_RATIO, voxel_ratio=VOXEL_RATIO,
    min_points_per_voxel=MIN_POINTS_PER_VOXEL, seed=SEED,
)
stages = pd.DataFrame({
    "etapa": ["nube de VGGT", "recorte esférico", "sin mesa", "componente principal"],
    "puntos": [clean_report[k] for k in ("input", "after_crop", "after_table", "after_component")],
})
stages["% del total"] = (100 * stages["puntos"] / stages["puntos"].iloc[0]).round(1)
print("Mesa detectada:", clean_report["table_plane_found"])
display(stages)

In [ ]:
from recon3d.measurements import MEASUREMENTS, NOT_AUTOMATIC, SCALE_BY
from recon3d.pointcloud import angular_coverage, dimensional_errors, object_frame

object_points = points[mask]
coords, rotation, origin = object_frame(object_points, clean_report["up"], seed=SEED)
scale, rows = dimensional_errors(coords, MEASUREMENTS[OBJECT], SCALE_BY[OBJECT])
measurements_df = pd.DataFrame(rows)
center_xy = ((np.array(clean_report["center"]) - origin) @ rotation.T)[:2]
coverage = angular_coverage(coords, center_xy)
evaluation = measurements_df[measurements_df["role"] == "evaluación"]

print(f"Factor de escala: {scale:.3f} mm por unidad de VGGT (medida: {SCALE_BY[OBJECT]})")
print(f"Cobertura angular: {coverage} % de la vuelta alrededor del objeto")
if OBJECT in NOT_AUTOMATIC:
    print("No evaluadas automáticamente:", NOT_AUTOMATIC[OBJECT])
print(f"Error absoluto medio de las medidas de evaluación: {evaluation['error_pct'].abs().mean():.2f} %")
display(measurements_df)

In [ ]:
coords_mm = coords * scale
low, high = np.percentile(coords_mm, 1, axis=0), np.percentile(coords_mm, 99, axis=0)
rng = np.random.default_rng(SEED)
keep = rng.choice(len(coords_mm), size=min(60000, len(coords_mm)), replace=False)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (i, j), label in zip(axes, [(0, 1), (0, 2), (1, 2)], ["x-y (planta)", "x-z (alzado)", "y-z (perfil)"]):
    ax.scatter(coords_mm[keep, i], coords_mm[keep, j], c=colors[mask][keep] / 255, s=0.3)
    ax.add_patch(plt.Rectangle((low[i], low[j]), high[i] - low[i], high[j] - low[j],
                               fill=False, color="red", linewidth=1.5))
    ax.set_xlabel("mm")
    ax.set_ylabel("mm")
    ax.set_aspect("equal")
    ax.set_title(label)
fig.suptitle(f"{OBJECT} con VGGT, escalado: caja envolvente {np.round(high - low, 1)} mm")
plt.show()

In [ ]:
import plotly.graph_objects as go

sample = rng.choice(len(coords_mm), size=min(40000, len(coords_mm)), replace=False)
rgb = colors[mask][sample]
fig = go.Figure(go.Scatter3d(
    x=coords_mm[sample, 0], y=coords_mm[sample, 1], z=coords_mm[sample, 2], mode="markers",
    marker=dict(size=1.5, color=[f"rgb({r},{g},{b})" for r, g, b in rgb]),
))
fig.update_layout(height=650, scene=dict(aspectmode="data", xaxis_title="mm",
                                         yaxis_title="mm", zaxis_title="mm"),
                  title=f"Nube de {OBJECT} con VGGT en mm (muestra de {len(sample):,} puntos)")
fig.show()

## 6. Guardar resultados

En `results/<objeto>/vggt/` de Drive:

- `object_clean_mm.ply`: nube del objeto limpia y escalada (mm).
- `cameras.npz`: extrínsecos, intrínsecos y nombres de las fotos usadas.
- `measurements.csv`, `cleaning.csv` y `meta.json` (con tiempo y memoria de GPU).

In [ ]:
from recon3d.pointcloud import write_ply

run_dir = paths.run_dir(OBJECT, METHOD)
run_dir.mkdir(parents=True, exist_ok=True)
write_ply(run_dir / "object_clean_mm.ply", coords_mm, colors=colors[mask])
np.savez(run_dir / "cameras.npz", names=np.array([p.name for p in frames]),
         extrinsic=extrinsic, intrinsic=intrinsic)
measurements_df.to_csv(run_dir / "measurements.csv", index=False)
stages.to_csv(run_dir / "cleaning.csv", index=False)

meta_path = write_meta(
    run_dir,
    object_name=OBJECT,
    method=METHOD,
    n_images=len(images),
    n_registered=len(frames),   # VGGT da cámara a todas las fotos que recibe
    runtime_s=time.perf_counter() - start_total,
    seed=SEED,
    scale_factor=float(scale),
    extra={
        "stage": "IA (VGGT)",
        "status": "ok",
        "vggt_commit": VGGT_COMMIT,
        "checkpoint": "facebook/VGGT-1B",
        "precision": str(dtype),
        "n_frames": len(frames),
        "n_images_used": len(frames),
        "frames": [p.name for p in frames],
        "blurry_images": blurry,
        "conf_percentile": CONF_PERCENTILE,
        "model_load_s": round(load_s, 1),
        "inference_s": round(inference_s, 2),
        "peak_gpu_gb": round(peak_gpu_gb, 2),
        "focal": {"fx_ref": round(fx_ref, 2), "fy_ref": round(fy_ref, 2),
                  "fx_vggt": round(float(fx_vggt), 2), "fy_vggt": round(float(fy_vggt), 2)},
        "cleaning": {"crop_ratio": CROP_RATIO, "plane_ratio": PLANE_RATIO,
                     "voxel_ratio": VOXEL_RATIO, "min_points_per_voxel": MIN_POINTS_PER_VOXEL,
                     **{k: v for k, v in clean_report.items() if k not in ("center", "up")}},
        "n_object_points": int(mask.sum()),
        "scale_by": SCALE_BY[OBJECT],
        "bounding_box_mm": np.round(high - low, 2).tolist(),
        "measurements": rows,
        "mean_abs_error_pct": round(float(evaluation["error_pct"].abs().mean()), 2),
        "angular_coverage_pct": coverage,
    },
)
for path in sorted(run_dir.iterdir()):
    print(f"{path.stat().st_size / 1e6:8.2f} MB  {path.name}")

## Siguiente paso

- Descargar el `.ipynb` ejecutado como `04_vggt_<objeto>.ipynb` en
  `notebooks/runs/` y `results/<objeto>/vggt` como zip en `results/<objeto>/`.
- Repetir con los cinco objetos cambiando `OBJECT`. A diferencia de COLMAP,
  VGGT siempre devuelve una reconstrucción: el colador y el vaso son los casos
  que más interesa comparar.
- Si aparecen valores no finitos (NaN), repetir con `PRECISION = "float32"` y,
  si falta memoria, bajar `N_FRAMES` a 40.